# TP2 — Partie 4 : Caractérisation des segments et recommandations
**TAISS 2026 — Filière F1 (Data Science)**

Entrées : `rfm_clusters.parquet` (Partie 3) + `transactions_clean.parquet` (Partie 1).
Sorties : le **tableau de synthèse** exigé par le brief, les figures de restitution, et les
recommandations marketing par segment.

Objectif : passer de « cluster 0, 1, 2, 3 » à des segments **nommés, chiffrés et actionnables**.
C'est cette partie que le marketing lira — les centroïdes ne parlent qu'aux data scientists.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.float_format", lambda v: f"{v:,.1f}")
sns.set_theme(style="whitegrid")

PROC   = Path("../data/processed")
FIG    = Path("../figures");  FIG.mkdir(parents=True, exist_ok=True)
REPORT = Path("../report");   REPORT.mkdir(parents=True, exist_ok=True)

rfm = pd.read_parquet(PROC / "rfm_clusters.parquet")
tx  = pd.read_parquet(PROC / "transactions_clean.parquet")
tx  = tx.merge(rfm[["CustomerID", "cluster"]], on="CustomerID")

print(f"{len(rfm):,} clients | {len(tx):,} transactions | {rfm.cluster.nunique()} segments")

## 4.1 Profil statistique de chaque cluster

On agrège R, F, M par cluster — moyenne **et médiane**. L'écart entre les deux révèle
l'hétérogénéité interne : un segment dont la moyenne dépasse largement la médiane contient
quelques clients extrêmes qui tirent le profil vers le haut.

In [ ]:
profil = rfm.groupby("cluster").agg(
    effectif     = ("CustomerID", "size"),
    R_moy        = ("Recence", "mean"),
    R_med        = ("Recence", "median"),
    F_moy        = ("Frequence", "mean"),
    F_med        = ("Frequence", "median"),
    M_moy        = ("Montant", "mean"),
    M_med        = ("Montant", "median"),
    panier_moyen = ("PanierMoyen", "mean"),
    anciennete   = ("Anciennete", "mean"),
)

profil["CA"]        = rfm.groupby("cluster").Montant.sum()
profil["pct_CA"]    = (100 * profil.CA / rfm.Montant.sum()).round(1)
profil["pct_clients"] = (100 * profil.effectif / len(rfm)).round(1)
profil = profil.sort_values("pct_CA", ascending=False)
display(profil.round(1))

## 4.2 Nommage des segments

**Règle importante** : les numéros de cluster attribués par K-means sont **arbitraires** et
changent d'une exécution à l'autre si la graine change. On ne code donc jamais
« cluster 1 = Champions » en dur — on nomme à partir des **valeurs** R/F/M relatives à la
médiane de la base.

Les quatre noms retenus suivent la terminologie RFM standard, immédiatement comprise en
marketing :

| Nom | Signature RFM | Enjeu |
|---|---|---|
| **Champions** | R faible, F élevée, M élevé | les retenir — ils font l'essentiel du CA |
| **À risque** | R élevée, F et M corrects | les relancer avant qu'ils ne partent |
| **Nouveaux / Prometteurs** | R faible, F faible, ancienneté courte | les faire monter en gamme |
| **Perdus / Dormants** | R très élevée, F = 1, M faible | les réactiver à coût maîtrisé, ou renoncer |

In [ ]:
def nommer(profil):
    """Nomme les 4 clusters par leur position RELATIVE, pas par un seuil fixe.
    Un seuil (ex. la médiane) est fragile : un cluster situé pile dessus bascule
    d'un nom à l'autre au hasard des arrondis."""
    restants = list(profil.index)
    noms = {}

    # 1. Perdus / Dormants : la plus forte récence (parti depuis le plus longtemps)
    c = profil.loc[restants].R_moy.idxmax()
    noms[c] = "Perdus / Dormants"; restants.remove(c)

    # 2. Champions : parmi les restants, celui qui dépense le plus
    c = profil.loc[restants].M_moy.idxmax()
    noms[c] = "Champions"; restants.remove(c)

    # 3. Des deux derniers : le plus ancien achat = À risque, l'autre = Nouveaux
    c = profil.loc[restants].R_moy.idxmax()
    noms[c] = "À risque"; restants.remove(c)
    noms[restants[0]] = "Nouveaux / Prometteurs"

    return noms


NOMS = nommer(profil)
print("Correspondance cluster → segment :")
for c, n in sorted(NOMS.items()):
    print(f"  cluster {c} → {n}")

rfm["segment"]    = rfm.cluster.map(NOMS)
tx["segment"]     = tx.cluster.map(NOMS)
profil["segment"] = profil.index.map(NOMS)

# garde-fou : si deux clusters recevaient le même nom, tout le reste du notebook
# (radar, gains chiffrés, tableau de synthèse) deviendrait faux silencieusement
assert profil.segment.nunique() == len(profil), "Deux clusters portent le même nom"

> ⚠️ **Vérifie que l'attribution automatique est cohérente** avec les profils affichés en 4.1.
> La règle ci-dessus suppose quatre profils bien contrastés ; si deux clusters recevaient le
> même nom, ajuste manuellement le dictionnaire `NOMS` en le justifiant.

## 4.3 Enrichissement : pays et produits dominants

Le brief demande explicitement une colonne « Top pays / produits » dans le tableau de synthèse.
On ajoute aussi la **diversité d'achat** (nombre de références distinctes par client) : c'est
un bon révélateur du profil grossiste vs particulier.

In [ ]:
def enrichir(g):
    ca = g.Amount.sum()
    pays = g.groupby("Country").Amount.sum().nlargest(2)
    prod = g.groupby("Description").Amount.sum().nlargest(3)
    return {
        "top_pays":     ", ".join(f"{p} ({100*v/ca:.0f} %)" for p, v in pays.items()),
        "part_UK":      100 * g.loc[g.Country == "United Kingdom", "Amount"].sum() / ca,
        "top_produits": " · ".join(str(p).title()[:28] for p in prod.index),
        "refs_par_client":     g.groupby("CustomerID").StockCode.nunique().mean(),
        "articles_par_client": g.groupby("CustomerID").Quantity.sum().mean(),
    }

# boucle explicite plutôt que groupby.apply : compatible avec toutes les versions de pandas
enrichi = pd.DataFrame({c: enrichir(g) for c, g in tx.groupby("cluster")}).T
enrichi.index.name = "cluster"
display(enrichi)

## 4.4 📋 Tableau de synthèse (livrable obligatoire)

C'est **le** livrable exigé par la section 5 du brief. À reprendre tel quel dans le rapport.

In [ ]:
synthese = pd.DataFrame({
    "Segment":            profil.segment,
    "Effectif":           profil.effectif,
    "% clients":          profil.pct_clients,
    "% CA":               profil.pct_CA,
    "Récence moy. (j)":   profil.R_moy.round(0),
    "Fréquence moy.":     profil.F_moy.round(1),
    "Montant moy. (£)":   profil.M_moy.round(0),
    "Panier moyen (£)":   profil.panier_moyen.round(0),
    "Top pays":           enrichi.top_pays,
    "Top produits":       enrichi.top_produits,
}).sort_values("% CA", ascending=False).reset_index(drop=True)

display(synthese)

synthese.to_csv(REPORT / "tableau_synthese.csv", index=False)
with open(REPORT / "tableau_synthese.md", "w", encoding="utf-8") as f:
    f.write(synthese.to_markdown(index=False))
print("✅ Exporté : report/tableau_synthese.csv et .md")

## 4.5 Figures de restitution

Trois visuels suffisent pour une soutenance. Le premier est le plus parlant : il montre le
**décalage entre poids en clients et poids en CA**.

In [ ]:
ordre = synthese.Segment.tolist()
pal = dict(zip(ordre, sns.color_palette("Set2", len(ordre))))

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
comp = synthese.set_index("Segment")[["% clients", "% CA"]]
comp.plot(kind="barh", ax=ax[0], color=["#B0B0B0", "#C44E52"])
ax[0].set_title("Poids en clients vs poids en chiffre d'affaires")
ax[0].set_xlabel("%"); ax[0].invert_yaxis()

ax[1].pie(synthese["% CA"], labels=synthese.Segment, autopct="%1.1f%%",
          colors=[pal[s] for s in ordre], startangle=90,
          wedgeprops={"edgecolor": "white", "linewidth": 2})
ax[1].set_title("Répartition du chiffre d'affaires")

plt.tight_layout(); plt.savefig(FIG / "04_poids_segments.png", dpi=150); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (col, titre) in zip(axes, [("Recence", "Récence (jours)"),
                                   ("Frequence", "Fréquence (commandes)"),
                                   ("Montant", "Montant (£)")]):
    sns.boxplot(data=rfm, x="segment", y=col, order=ordre,
                hue="segment", palette=pal, legend=False,
                ax=ax, showfliers=False)
    ax.set_title(titre); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=25)
    if col in ("Frequence", "Montant"):
        ax.set_yscale("log")
plt.tight_layout(); plt.savefig(FIG / "04_distributions_segments.png", dpi=150); plt.show()

### Radar des profils
Les valeurs sont normalisées entre 0 et 1 pour être comparables. **La Récence est inversée**
(1 = très récent) afin que « plus loin du centre = meilleur » sur les trois axes.

In [ ]:
radar = profil.set_index("segment")[["R_moy", "F_moy", "M_moy"]].copy()
radar["R_moy"] = radar.R_moy.max() - radar.R_moy          # inversion : haut = récent
norm = (radar - radar.min()) / (radar.max() - radar.min())

# .loc[seg] renvoie une Series tant que les noms de segments sont uniques
# (garanti par l'assert de la cellule de nommage)

axes_lbl = ["Récence\n(inversée)", "Fréquence", "Montant"]
ang = np.linspace(0, 2*np.pi, len(axes_lbl), endpoint=False).tolist(); ang += ang[:1]

fig, ax = plt.subplots(figsize=(6.5, 6.5), subplot_kw={"polar": True})
for seg in ordre:
    v = norm.loc[seg].to_numpy().ravel().tolist(); v += v[:1]
    ax.plot(ang, v, "o-", lw=2, label=seg, color=pal[seg])
    ax.fill(ang, v, alpha=.12, color=pal[seg])
ax.set_xticks(ang[:-1]); ax.set_xticklabels(axes_lbl)
ax.set_yticklabels([]); ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1))
ax.set_title("Profil RFM par segment", pad=20)
plt.tight_layout(); plt.savefig(FIG / "04_radar_segments.png", dpi=150); plt.show()

## 4.6 Recommandations marketing

Chaque recommandation suit la même structure : **constat chiffré → action → indicateur de
succès**. Une action sans métrique de suivi n'est pas une recommandation, c'est une intention.

In [ ]:
reco = pd.DataFrame([
    {"Segment": "Champions",
     "Constat": "~20 % des clients, ~73 % du CA, achat il y a moins d'un mois",
     "Action": "Programme de fidélité premium : accès anticipé aux nouveautés, contact commercial dédié, conditions de gros négociées",
     "Indicateur": "taux de rétention à 12 mois, CA moyen par client",
     "Priorité": 1},
    {"Segment": "À risque",
     "Constat": "Bons acheteurs historiques mais silencieux depuis ~7 mois",
     "Action": "Campagne de réactivation ciblée : email personnalisé sur leurs références habituelles + offre de retour limitée dans le temps",
     "Indicateur": "taux de réachat à 60 jours",
     "Priorité": 2},
    {"Segment": "Nouveaux / Prometteurs",
     "Constat": "Achat récent mais 3 commandes en moyenne, panier modeste",
     "Action": "Séquence d'onboarding : recommandations croisées, remise sur la 2ᵉ commande, mise en avant du catalogue élargi",
     "Indicateur": "passage à 3+ commandes sous 6 mois",
     "Priorité": 2},
    {"Segment": "Perdus / Dormants",
     "Constat": "Une seule commande, il y a plus d'un an, faible montant",
     "Action": "Campagne de win-back à coût maîtrisé (email de masse, pas de remise agressive) ; sortie de la base active si sans réponse",
     "Indicateur": "taux de réactivation ; coût par client réactivé",
     "Priorité": 3},
])

display(reco)
reco.to_csv(REPORT / "recommandations.csv", index=False)
with open(REPORT / "recommandations.md", "w", encoding="utf-8") as f:
    f.write(reco.to_markdown(index=False))

### Chiffrer l'enjeu — l'argument qui convainc une direction

Une recommandation prend du poids quand on estime ce qu'elle rapporte. On calcule ici le CA
« à risque » (celui des clients qui s'éloignent) et le potentiel de montée en gamme.

In [ ]:
ca_total = rfm.Montant.sum()

for seg in ordre:
    g = rfm[rfm.segment == seg]
    print(f"{seg:24s} : {len(g):>5,} clients | CA {g.Montant.sum():>12,.0f} £ "
          f"({100*g.Montant.sum()/ca_total:>4.1f} %) | CA/client {g.Montant.mean():>8,.0f} £")

risque = rfm[rfm.segment == "À risque"]
champ  = rfm[rfm.segment == "Champions"]
promet = rfm[rfm.segment == "Nouveaux / Prometteurs"]

print(f"\n▸ CA exposé sur le segment 'À risque' : {risque.Montant.sum():,.0f} £")
print(f"▸ Si 20 % des 'À risque' remontaient au niveau des Champions :")
print(f"   gain ≈ {0.2 * len(risque) * (champ.Montant.mean() - risque.Montant.mean()):,.0f} £")
print(f"▸ Si 10 % des 'Prometteurs' atteignaient le niveau 'À risque' :")
print(f"   gain ≈ {0.1 * len(promet) * (risque.Montant.mean() - promet.Montant.mean()):,.0f} £")

> ⚠️ **Précaution à énoncer à l'oral** : ces montants sont des **ordres de grandeur
> illustratifs**, pas des prévisions. Ils supposent qu'un client réactivé se comporte comme la
> moyenne de son segment cible, ce que rien ne garantit. Ils servent à hiérarchiser les
> priorités, pas à construire un budget. Un modèle de CLV prédictif (extension `05a`) donnerait
> une estimation défendable.

## 4.7 Contrôles de cohérence

Deux vérifications avant de conclure — elles évitent les mauvaises surprises en soutenance.

In [ ]:
# 1. Les segments recoupent-ils la grille RFM par quintiles de la Partie 2 ?
print("Score RFM moyen (somme R+F+M sur 15) par segment :")
display(rfm.groupby("segment").RFM_somme.mean().sort_values(ascending=False).round(2))

# 2. Y a-t-il un biais géographique ?
print("\nRépartition des segments dans les 5 premiers pays (% de la ligne) :")
top_pays = tx.groupby("Country").Amount.sum().nlargest(5).index
display((100 * pd.crosstab(tx[tx.Country.isin(top_pays)].Country,
                           tx[tx.Country.isin(top_pays)].segment,
                           normalize="index")).round(1))

> **À vérifier** : les scores RFM moyens doivent décroître dans le même ordre que le % de CA.
> Si ce n'est pas le cas, le nommage ou le clustering mérite un second regard.
>
> Sur la géographie : ~91 % du CA vient du Royaume-Uni. Une différence de répartition des
> segments entre pays reflète surtout la petite taille des échantillons étrangers — **ne pas
> sur-interpréter**, et le mentionner dans les limites (Partie 5).

## 4.8 Export final

In [ ]:
rfm.to_parquet(PROC / "rfm_segments.parquet", index=False)
rfm[["CustomerID", "segment", "Recence", "Frequence", "Montant"]].to_csv(
    PROC / "clients_segmentes.csv", index=False)

print("✅ Livrables Partie 4 :")
print("   report/tableau_synthese.csv + .md   ← livrable obligatoire")
print("   report/recommandations.csv + .md")
print("   figures/04_*.png (3 figures)")
print("   data/processed/rfm_segments.parquet")

## Mini-résumé (réutilisable dans le rapport)

> Les quatre segments sont nommés à partir de leurs valeurs R/F/M — jamais de leur numéro de
> cluster, arbitraire. Les **Champions** (1 184 clients, 20 % de la base) achètent en moyenne
> 19 fois pour 10 581 £ avec une récence de 28 jours : ils concentrent **73 % du chiffre
> d'affaires** et commandent 208 références distinctes en moyenne, un profil de grossiste. Les
> clients **À risque** (1 454, 17 % du CA) ont un historique solide — 5 commandes, 1 974 £ — mais
> n'ont plus acheté depuis 228 jours : c'est le gisement de réactivation prioritaire. Les
> **Nouveaux / Prometteurs** (1 246, 6 % du CA) sont récents (28 jours) mais peu engagés
> (3 commandes, 841 £) et récents dans la base (278 jours d'ancienneté contre 574 pour les
> clients à risque). Les **Perdus / Dormants** (1 968, soit un tiers des clients) n'ont passé
> qu'une commande de 317 £ il y a plus d'un an et ne pèsent que 3,7 % du CA. Le déséquilibre
> est frappant : un cinquième des clients porte près des trois quarts du chiffre d'affaires,
> ce qui justifie de concentrer l'effort de rétention plutôt que l'acquisition indifférenciée.

**Livrable obligatoire produit** : tableau de synthèse ✅
**Pour la Partie 5** : concentration extrême du CA (dépendance à ~1 200 clients), biais
géographique (91 % UK), et caractère illustratif des gains chiffrés.